<a href="https://colab.research.google.com/github/aleksandrovd2-dev/ACAS_methodology/blob/main/ACAS_python_library_(eng).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
from dataclasses import dataclass
from typing import Tuple

@dataclass
class ACASConfig:
    """
    Global settings and constants for the ACAS model.
    """
    R1: float = 0.5       # Fixed starting rating for a new Customer
    PM: float = 3.0       # Median maturation point (months)
    k: float = 2.0        # Steepness coefficient - Adapted Hill function
    MAX_PL: float = 12.0  # Rolling window limit in months

class ACASScorer:
    """
    Main class for ACAS calculation.
    """
    def __init__(self, config: ACASConfig = ACASConfig()):
        self.config = config

    def calculate(
        self,
        pl: float,
        pa: float,
        vi: float,
        vp: float,
        vr: float,
        ai: float,
        ap: float,
        ar: float
    ) -> Tuple[float, str]:
        """
        Calculates the Customer's rating and returns percentage and text status.
        """
        # 1. Rolling Window Limit
        # Protection against zero PL to prevent division by zero
        safe_pl = max(min(pl, self.config.MAX_PL), 0.1)
        safe_pa = max(pa, 0.1)

        # 2. Weighted Effectiveness (WE)
        # Protection against division by zero if no accounts
        if vi == 0 or ai == 0:
            we = 0.0
        else:
            we = 0.5 * (((vp - vr) / vi) + ((ap - ar) / ai))
            we = max(0.0, min(we, 1.0)) # Metric strictly lies in the range from 0 to 1

        # 3. Activity Gap (AG)
        ag = 1.0 - (safe_pa / safe_pl)
        ag = max(0.0, ag) # Protection against negative values

        # 4. Zero Pressure Coefficient (ZP)
        # Uses quantitative document turnover metrics
        zp = (vi - vp + vr) / safe_pa
        zp = max(0.0, zp)

        # 5. Loss Index (WI)
        wi = 0.5 * (ag + zp)

        # 6. Hard Calculated Rating (R2)
        # If WI > 1 (huge zero pressure), the rating drops to 0
        penalty = max(0.0, 1.0 - wi)
        r2 = we * penalty

        # 7. Dynamic Smoothing (w_pl) - Adapted Hill function
        w_pl = 1.0 / (1.0 + (safe_pl / self.config.PM) ** self.config.k)

        # 8. Final Synergistic Rating (Racas)
        racas_score = w_pl * self.config.R1 + (1.0 - w_pl) * r2

        # Convert to percentage
        racas_percent = round(racas_score * 100, 2)

        # 9. Determine status by matrix
        status = self._get_status(racas_percent)

        return racas_percent, status

    def _get_status(self, score: float) -> str:
        """
        Translates percentage rating into business status.
        """
        if score < 0.0:
            return "Critical"
        elif 0.0 == score <= 0.99:
            return "Zero"
        elif 1.0 <= score <= 5.0:
            return "Very Low"
        elif 5.01 <= score <= 25.0:
            return "Low"
        elif 25.01 <= score <= 65.0:
            return "Medium"
        elif 65.01 <= score <= 95.0:
            return "High"
        else:
            return "Very High"

# ==========================================
# Testing block with examples from the article
# ==========================================
if __name__ == "__main__":
    scorer = ACASScorer()

    print("=== ACAS Methodology Testing ===")

    # Customer #1: Promising newcomer
    score1, status1 = scorer.calculate(
        pl=1.5, pa=1.0,
        vi=2.0, vp=2.0, vr=0.0,
        ai=1.0, ap=0.9, ar=0.0
    )
    print(f"Customer #1 (Newcomer): Rating {score1}% | Status: {status1}")

    # Customer #2: Old "time burner"
    score2, status2 = scorer.calculate(
        pl=12.0, pa=2.0,
        vi=50.0, vp=2.0, vr=0.0,
        ai=1.0, ap=0.2, ar=0.0
    )
    print(f"Customer #2 (Passive): Rating {score2}% | Status: {status2}")

    # Customer #3: Reliable regular partner
    score3, status3 = scorer.calculate(
        pl=6.0, pa=6.0,
        vi=12.0, vp=12.0, vr=0.0,
        ai=1.0, ap=1.0, ar=0.0
    )
    print(f"Customer #3 (Reliable): Rating {score3}% | Status: {status3}")

=== ACAS Methodology Testing ===
Customer #1 (Newcomer): Rating 55.83% | Status: Medium
Customer #2 (Passive): Rating 2.94% | Status: Very Low
Customer #3 (Reliable): Rating 90.0% | Status: High
